<a href="https://colab.research.google.com/github/felipetavaressilvaoliveira-netizen/senacai/blob/main/analise_de_som_desafio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import kagglehub
import os

# 1. DOWNLOAD
path = kagglehub.dataset_download("soumendraprasad/musical-instruments-sound-dataset")
# The actual audio files for training are in a subfolder structure, typically:
# <download_path>/Train_submission/Train_submission/
# The metadata (labels) are in:
# <download_path>/Train_submission/Metadata_Train.csv

# Print path information for debugging
print("Dataset downloaded to:", path)
print("Content of download directory:", os.listdir(path))
print("Content of Train_submission directory:", os.listdir(os.path.join(path, "Train_submission")))
print(os.listdir(os.path.join(path, "Train_submission")))


Using Colab cache for faster access to the 'musical-instruments-sound-dataset' dataset.
Dataset downloaded to: /kaggle/input/musical-instruments-sound-dataset
Content of download directory: ['Test_submission', 'Train_submission', 'Metadata_Train.csv', 'Metadata_Test.csv']
Content of Train_submission directory: ['Train_submission']


In [23]:
print(os.listdir(path))

['Test_submission', 'Train_submission', 'Metadata_Train.csv', 'Metadata_Test.csv']


In [32]:
import numpy as np
import librosa
import librosa.display
import tensorflow as tf
from sklearn.model_selection import train_test_split
import os
import pandas as pd # Import pandas for reading metadata

# Configurações de Memória
IMG_SIZE = (224, 224)
CLASSES = ['Sound_Guitar', 'Sound_Drum', 'Sound_Violin', 'Sound_Piano'] # Corrected class labels to match metadata
MAX_FILES_PER_CLASS = 60  # Reduzimos aqui para garantir que a RAM aguente

def audio_to_spectrogram(audio_path):
    # Carrega apenas 2 segundos para economizar muita RAM
    y, sr = librosa.load(audio_path, duration=2)

    # Handle cases where audio is too short or silent
    if len(y) == 0 or np.all(y == 0):
        # Return an array of zeros with the expected shape (height, width, channels)
        # The .numpy() conversion will happen outside for consistency with other data
        return tf.zeros(IMG_SIZE + (3,), dtype=tf.float32)

    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    # Convert to dB, adding a small constant to prevent log(0) and clipping values below top_db
    S_dB = librosa.power_to_db(S + 1e-8, ref=np.max, top_db=80.0)

    # Normalize S_dB to a 0-1 range
    min_val = np.min(S_dB)
    max_val = np.max(S_dB)

    if max_val == min_val: # If the spectrogram is flat (e.g., all silence after dB conversion)
        normalized_S_dB = np.zeros_like(S_dB)
    else:
        normalized_S_dB = (S_dB - min_val) / (max_val - min_val)

    # Redimensiona e converte para RGB (3 canais) para o MobileNet
    img = tf.image.resize(np.expand_dims(normalized_S_dB, -1), IMG_SIZE)
    img = tf.image.grayscale_to_rgb(img)
    return img

# Define the base paths for audio files and metadata
# 'path' variable is expected to be available from the previous cell's execution.
# Based on typical Kaggle dataset structure and previous outputs:
audio_files_base_path = os.path.join(path, "Train_submission", "Train_submission")
metadata_file_path = os.path.join(path, "Metadata_Train.csv") # Corrected path for metadata file

x_data, y_data = [], []
label_to_int = {label: i for i, label in enumerate(CLASSES)}

# Load metadata to get file names and their corresponding labels
if not os.path.exists(metadata_file_path):
    print(f"Error: Metadata file not found at '{metadata_file_path}'. Cannot load data.")
else:
    metadata_df = pd.read_csv(metadata_file_path)
    print(f"Metadata loaded from: {metadata_file_path}")

    # Iterate through each class to load a limited number of files
    for label_str in CLASSES:
        class_df = metadata_df[metadata_df['Class'] == label_str].head(MAX_FILES_PER_CLASS)
        print(f"Lendo {len(class_df)} arquivos para a classe '{label_str}'...")

        for index, row in class_df.iterrows():
            file_name = row['FileName']
            full_audio_path = os.path.join(audio_files_base_path, file_name)

            if not os.path.exists(full_audio_path):
                print(f"Warning: Audio file not found at '{full_audio_path}'. Skipping.")
                continue

            try:
                spec = audio_to_spectrogram(full_audio_path)
                x_data.append(spec.numpy()) # Converte para numpy para economizar memória de GPU
                y_data.append(label_to_int[label_str])
            except Exception as e:
                print(f"Error processing {full_audio_path}: {e}. Skipping.")

X = np.array(x_data)
y = np.array(y_data)

if len(X) == 0:
    print("Error: No audio data was successfully loaded. Please check dataset paths and contents.")
    # Prevent ValueError from train_test_split with empty data
    X_train, X_test, y_train, y_test = np.array([]), np.array([]), np.array([]), np.array([])
else:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) # Added stratify

print(f"\nSucesso! {len(X)} imagens processadas.")
print(f"Formato do array X_train: {X_train.shape}")
print(f"Formato do array y_train: {y_train.shape}")
print(f"Formato do array X_test: {X_test.shape}")
print(f"Formato do array y_test: {y_test.shape}")


Metadata loaded from: /kaggle/input/musical-instruments-sound-dataset/Metadata_Train.csv
Lendo 60 arquivos para a classe 'Sound_Guitar'...
Lendo 60 arquivos para a classe 'Sound_Drum'...
Lendo 60 arquivos para a classe 'Sound_Violin'...
Lendo 60 arquivos para a classe 'Sound_Piano'...

Sucesso! 240 imagens processadas.
Formato do array X_train: (192, 224, 224, 3)
Formato do array y_train: (192,)
Formato do array X_test: (48, 224, 224, 3)
Formato do array y_test: (48,)


In [33]:
from tensorflow.keras import layers, models, applications

# Criando o modelo baseado em MobileNetV2 - nesse momento estamos usando o conceito MobileNetV2
base_model = applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(CLASSES), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("Iniciando treinamento final...")
history = model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test), batch_size=16)

Iniciando treinamento final...
Epoch 1/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 18s 924ms/step - accuracy: 0.3490 - loss: 1.4555 - val_accuracy: 0.5417 - val_loss: 1.0275
Epoch 2/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 10s 786ms/step - accuracy: 0.6042 - loss: 0.8995 - val_accuracy: 0.5833 - val_loss: 0.9100
Epoch 3/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 9s 777ms/step - accuracy: 0.6719 - loss: 0.6843 - val_accuracy: 0.5000 - val_loss: 0.8305
Epoch 4/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 9s 778ms/step - accuracy: 0.6771 - loss: 0.6035 - val_accuracy: 0.5417 - val_loss: 0.8287
Epoch 5/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 9s 644ms/step - accuracy: 0.7292 - loss: 0.5324 - val_accuracy: 0.5208 - val_loss: 0.8412
Epoch 6/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 10s 870ms/step - accuracy: 0.7344 - loss: 0.5228 - val_accuracy: 0.5208 - val_loss: 0.8038
Epoch 7/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 11s 945ms/step - accuracy: 0.7500 - loss: 0.4865 - val_accuracy: 0.4792 - val_loss: 0.8541
Epoch 8/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 9s 739ms/step - accuracy: 0.724

In [38]:
import librosa.display

# Install resampy if not already installed
try:
    import resampy
except ImportError:
    print("Installing resampy...")
    !pip install resampy
    import resampy

filename="/kaggle/input/musical-instruments-sound-dataset/Test_submission/Test_submission/Sad-Violin-J-Slow-www.fesliyanstudios.com.wav"
audio,sample_rate=librosa.load(filename)
mfccs_features=librosa.feature.mfcc(y=audio,sr=sample_rate,n_mfcc=40)
mfccs_scaled_features =np.mean(mfccs_features.T,axis=0)

print(mfccs_scaled_features)
mfccs_scaled_features=mfccs_scaled_features.reshape(1,-1)
print(mfccs_scaled_features)

# The model was trained with spectrograms, not MFCCs. This prediction will not be accurate.
# Also, the LabelEncoder (LE) is not defined in the current scope.
# To fix this, you would need to:
# 1. Adapt the audio_to_spectrogram function for single inference.
# 2. Use that function to transform the test audio.
# 3. Ensure LE is loaded/created with the same classes as during training.
# For now, let's just make sure the basic librosa loading works.

predicted_label = np.argmax(model.predict(mfccs_scaled_features), axis=-1)
print(predicted_label)

# prediction_class=LE.inverse_transform(predicted_label)
# prediction_class


[-3.7968219e+02 -9.6396893e-01 -4.3830162e+01 -2.3903069e+00
 -2.4881378e+01 -2.0166260e+01 -2.4435402e+01 -1.6559214e+01
 -1.2569412e+01 -3.6521142e+00 -1.0912807e+01 -8.9176359e+00
  1.2977679e+00  1.8845626e+00 -7.5638433e+00  5.1654639e+00
  5.5502267e+00  1.2195654e+00  2.6087849e+00  5.3130656e-01
  1.3297251e+01  1.4455271e+01 -8.8572222e-01 -5.3815711e-01
 -4.7728381e-01  5.6049199e+00 -5.4910231e+00  7.8543174e-01
  5.1636953e+00 -1.5319449e+01  2.3116820e+00  1.3546501e+01
 -1.1675764e+00 -7.6847281e+00 -2.0583491e+00 -8.3604298e+00
 -2.4785774e+00  1.0146462e-01 -4.0663040e-01 -4.1813641e+00]
[[-3.7968219e+02 -9.6396893e-01 -4.3830162e+01 -2.3903069e+00
  -2.4881378e+01 -2.0166260e+01 -2.4435402e+01 -1.6559214e+01
  -1.2569412e+01 -3.6521142e+00 -1.0912807e+01 -8.9176359e+00
   1.2977679e+00  1.8845626e+00 -7.5638433e+00  5.1654639e+00
   5.5502267e+00  1.2195654e+00  2.6087849e+00  5.3130656e-01
   1.3297251e+01  1.4455271e+01 -8.8572222e-01 -5.3815711e-01
  -4.7728381e-01 

ValueError: Exception encountered when calling Sequential.call().

[1mInvalid input shape for input Tensor("data:0", shape=(1, 40), dtype=float32) with name 'keras_tensor_794' and path ''. Expected shape (None, 224, 224, 3), but input has incompatible shape (1, 40)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(1, 40), dtype=float32)
  • training=False
  • mask=None
  • kwargs=<class 'inspect._empty'>